# 🧩 Абстрактные классы и протоколы: проектирование интерфейсов в Python

<div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Как заставить классы соблюдать контракт</h2>
  <p>В этом занятии мы разберём два мощных инструмента для определения интерфейсов в Python:</p>
  <ul>
    <li><strong>Абстрактные классы</strong> (abc.ABC, @abstractmethod) – явное наследование и контроль реализации</li>
    <li><strong>Протоколы</strong> (typing.Protocol) – структурная типизация («утиная типизация» на стероидах)</li>
  </ul>
  <p>Вы узнаете, как проектировать гибкие и типобезопасные иерархии классов.</p>
</div>

## 🎯 Цели лекции

- Понять, зачем нужны абстрактные классы и чем они отличаются от обычных
- Научиться создавать абстрактные классы с помощью модуля `abc`
- Разобраться с абстрактными методами, свойствами и их реализацией в наследниках
- Познакомиться с протоколами (`Protocol`) и структурной типизацией
- Сравнить подходы: наследование vs утиная типизация vs протоколы
- Увидеть практические примеры использования в реальных проектах

## 📚 План лекции

1. **Абстрактные классы: зачем и когда**
   - Проблема: как обязать наследников реализовать методы
   - Абстрактный класс как «шаблон» или «контракт»
   - Отличие от конкретных классов

2. **Модуль `abc`**
   - `ABC` – базовый класс для абстрактных классов
   - `@abstractmethod` – декоратор для абстрактных методов
   - `@abstractproperty` (устарел) и `@abstractmethod` для свойств

3. **Особенности абстрактных классов в Python**
   - Нельзя создать экземпляр абстрактного класса
   - Частичная реализация – класс может быть абстрактным, если не все методы переопределены
   - Абстрактные методы могут иметь реализацию (но вызываться через `super()`)

4. **Протоколы и структурная типизация**
   - Утиная типизация в Python: «если это крякает как утка…»
   - Проблема: статическая проверка типов (mypy) не знает о «кряканье»
   - `typing.Protocol` – определение интерфейса без наследования
   - `@runtime_checkable` – проверка протоколов во время выполнения

5. **Сравнение подходов**
   - Абстрактные классы: явное наследование, контроль иерархии
   - Протоколы: гибкость, совместимость с существующими классами
   - Когда что выбирать?

6. **Практические примеры**
   - Абстрактный класс `Shape` с методами `area`, `perimeter`
   - Протокол `Drawable` для разных типов объектов
   - Комбинирование подходов

7. **Частые ошибки и подводные камни**
8. **Итоги**

In [ ]:
# Проблема: как гарантировать, что все фигуры реализуют метод area?
class Square:
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

class Circle:
    def __init__(self, radius):
        self.radius = radius
    # забыли реализовать area() – код упадёт только в runtime
    # def area(self):
    #     return 3.14159 * self.radius ** 2

def print_area(shape):
    print(f"Площадь: {shape.area()}")  # AttributeError, если нет метода

c = Circle(5)
# print_area(c)  # Ошибка!

## 1. Абстрактные классы: зачем и когда

**Абстрактный класс** – это класс, который содержит хотя бы один абстрактный метод. Абстрактный метод объявлен, но не реализован. Наследники **обязаны** реализовать все абстрактные методы, иначе они сами останутся абстрактными.

### Зачем нужны абстрактные классы?
- **Документация интерфейса** – сразу видно, какие методы должны быть реализованы.
- **Контроль** – интерпретатор не даст создать экземпляр класса, если он не реализовал все абстрактные методы.
- **Единая точка расширения** – базовый класс определяет, как взаимодействовать с наследниками.

### Отличие от конкретных классов
- Конкретный класс можно инстанцировать, абстрактный – нет.
- Абстрактный класс может содержать как абстрактные, так и обычные методы (с реализацией).

In [ ]:
# Пример абстрактного класса через abc
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        """Вычисляет площадь фигуры."""
        pass

    @abstractmethod
    def perimeter(self):
        """Вычисляет периметр фигуры."""
        pass

    def description(self):
        """Обычный метод – может быть унаследован."""
        return f"Фигура с площадью {self.area()}"

# Попытка создать экземпляр абстрактного класса вызовет ошибку
# s = Shape()  # TypeError: Can't instantiate abstract class Shape with abstract methods area, perimeter

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(3, 4)
print(r.area())
print(r.perimeter())
print(r.description())

## 2. Модуль `abc` подробнее

- **`ABC`** – вспомогательный класс, от которого удобно наследоваться. Можно также мета-класс `ABCMeta`.
- **`@abstractmethod`** – делает метод абстрактным. Если класс содержит хотя бы один такой метод, он становится абстрактным.

### Абстрактные свойства
Для свойства используйте `@property` вместе с `@abstractmethod`.

In [ ]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    @property
    @abstractmethod
    def max_speed(self):
        pass

    @abstractmethod
    def start_engine(self):
        pass

class Car(Vehicle):
    @property
    def max_speed(self):
        return 220

    def start_engine(self):
        return "Др-др-др!"

c = Car()
print(c.max_speed)
print(c.start_engine())

In [ ]:
# Абстрактный метод может иметь реализацию (вызываемую через super())
class Base(ABC):
    @abstractmethod
    def greet(self):
        print("Hello from Base")

class Derived(Base):
    def greet(self):
        super().greet()
        print("Hello from Derived")

d = Derived()
d.greet()

## 3. Особенности абстрактных классов в Python

- **Частичная абстракция** – класс, в котором не все абстрактные методы переопределены, остаётся абстрактным.
- **Абстрактные методы могут иметь тело** – но это необычно; обычно их оставляют с `pass` или с документацией.
- **Комбинация с другими декораторами** – можно создавать абстрактные статические методы и методы класса.

### Пример: абстрактный класс с частичной реализацией

In [ ]:
class AbstractStream(ABC):
    @abstractmethod
    def read(self):
        pass

    @abstractmethod
    def write(self, data):
        pass

    def copy(self, other_stream):
        """Обычный метод, использующий абстрактные."""
        data = self.read()
        other_stream.write(data)

class FileStream(AbstractStream):
    def read(self):
        return "data from file"
    def write(self, data):
        print(f"Writing {data} to file")

# Класс, реализующий только read, останется абстрактным
class ReadOnlyStream(AbstractStream):
    def read(self):
        return "read only"
    # write не реализован → класс абстрактный, инстанцировать нельзя
# r = ReadOnlyStream()  # TypeError

## 4. Протоколы и структурная типизация

**Утиная типизация**: «Если это ходит как утка и крякает как утка, то это утка». Python не требует явного наследования – достаточно наличия нужных методов.

Но статические анализаторы (mypy, Pyright) не могут проверить утиную типизацию без подсказок.

**Протоколы** (`typing.Protocol`) – способ описать интерфейс, которому объект может соответствовать, **не наследуясь** от него. Это структурная типизация (static duck typing).

### Синтаксис

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> None:
        ...

class Circle:
    def draw(self) -> None:
        print("Рисуем круг")

class Square:
    def draw(self) -> None:
        print("Рисуем квадрат")

class Point:
    def plot(self):
        print("Точка")  # нет метода draw

def render(obj: Drawable) -> None:
    obj.draw()

render(Circle())   # OK
render(Square())   # OK
# render(Point())  # mypy: error: "Point" has no attribute "draw"

### Протоколы во время выполнения: `@runtime_checkable`

По умолчанию `isinstance(obj, Protocol)` не работает (протокол – только для статических проверок). Добавив `@runtime_checkable`, мы можем проверять соответствие протоколу через `isinstance()` (проверяется наличие методов).

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Serializable(Protocol):
    def serialize(self) -> str:
        ...

class User:
    def serialize(self) -> str:
        return "User data"

class Product:
    def to_json(self) -> str:
        return "{}"

u = User()
p = Product()

print(isinstance(u, Serializable))  # True (есть метод serialize)
print(isinstance(p, Serializable))  # False (нет serialize)

## 5. Сравнение подходов: абстрактные классы vs протоколы

| Характеристика | Абстрактный класс | Протокол |
|----------------|-------------------|-----------|
| Наследование | Явное (`class Dog(Animal)`) | Неявное (структурное) |
| Проверка в runtime | `isinstance` работает | Только с `@runtime_checkable` (проверяет наличие методов) |
| Статическая проверка (mypy) | Да | Да |
| Может содержать реализацию методов | Да | Нет (только сигнатуры) |
| Контроль создания экземпляров | Нельзя создать абстрактный класс | Можно создать любой, протокол – только интерфейс |
| Множественное наследование | Да (но осторожно) | Не применимо (это не класс) |
| Основное назначение | Иерархия классов с общей логикой | Гибкие интерфейсы для разных типов |

### Когда что выбирать?
- **Абстрактные классы** – когда у вас есть иерархия «is-a», общая реализация методов, или нужно хранить состояние.
- **Протоколы** – когда вы хотите определить интерфейс для независимых классов, не заставляя их наследоваться, или работаете с чужим кодом.

In [ ]:
# Пример: использование протокола для работы с разными объектами
from typing import Protocol

class CanQuack(Protocol):
    def quack(self) -> str:
        ...

class Duck:
    def quack(self) -> str:
        return "Quack!"

class Person:
    def quack(self) -> str:
        return "Imitating quack!"

class Dog:
    def bark(self) -> str:
        return "Woof!"

def make_it_quack(obj: CanQuack):
    print(obj.quack())

make_it_quack(Duck())      # OK
make_it_quack(Person())    # OK
# make_it_quack(Dog())     # mypy error

## 6. Практические примеры

### Пример 1: Абстрактный класс для хранилища данных

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Any

class DataStore(ABC):
    @abstractmethod
    def save(self, key: str, value: Any) -> None:
        pass

    @abstractmethod
    def load(self, key: str) -> Any:
        pass

    @abstractmethod
    def delete(self, key: str) -> None:
        pass

class InMemoryStore(DataStore):
    def __init__(self):
        self._data = {}

    def save(self, key: str, value: Any) -> None:
        self._data[key] = value

    def load(self, key: str) -> Any:
        return self._data.get(key)

    def delete(self, key: str) -> None:
        self._data.pop(key, None)

class FileStore(DataStore):
    def __init__(self, filename: str):
        self.filename = filename

    def save(self, key: str, value: Any) -> None:
        # реализация с записью в файл
        pass

    def load(self, key: str) -> Any:
        pass

    def delete(self, key: str) -> None:
        pass

store = InMemoryStore()
store.save("user", "Alice")
print(store.load("user"))

### Пример 2: Протокол для итерации (кастомный итератор)

In [ ]:
from typing import Protocol, Iterator

class Countdown:
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

class IterableProtocol(Protocol):
    def __iter__(self) -> Iterator:
        ...

def print_items(iterable: IterableProtocol):
    for item in iterable:
        print(item)

print_items(Countdown(3))
print_items([1, 2, 3])    # list тоже подходит
print_items("abc")        # строка подходит

### Пример 3: Комбинирование абстрактного класса и протокола
Иногда удобно: абстрактный класс определяет общую логику, а протокол – отдельный интерфейс для части функциональности.

In [ ]:
class JSONSerializable(Protocol):
    def to_json(self) -> str:
        ...

class Loggable(ABC):
    @abstractmethod
    def log(self, message: str) -> None:
        pass

class User(Loggable, JSONSerializable):
    def __init__(self, name: str):
        self.name = name

    def log(self, message: str) -> None:
        print(f"[USER] {message}")

    def to_json(self) -> str:
        return f'{{"name": "{self.name}"}}'

def save_json(obj: JSONSerializable, filename: str):
    with open(filename, "w") as f:
        f.write(obj.to_json())

u = User("Alice")
save_json(u, "user.json")

## 7. Частые ошибки и подводные камни

1. **Забыть унаследоваться от `ABC`** – тогда `@abstractmethod` не будет работать (класс не станет абстрактным, можно создать экземпляр).
2. **Вызов `super().__init__()` в абстрактном классе** – если в `__init__` есть абстрактные методы, они не будут реализованы к моменту вызова.
3. **Использование `@abstractmethod` со статическими методами** – возможно, но редко нужно.
4. **Смешивание протоколов с `isinstance` без `@runtime_checkable`** – будет всегда `False` (но mypy поймёт).
5. **Протокол с методами, имеющими реализацию по умолчанию** – протокол определяет только сигнатуры, реализацию не даёт.

### Пример ошибки: не наследован ABC

In [ ]:
from abc import abstractmethod

class Bad(ABC):  # если убрать ABC, класс не станет абстрактным
    @abstractmethod
    def foo(self):
        pass

b = Bad()  # Ошибка: Can't instantiate abstract class Bad – но только если есть ABC!
# Без ABC это бы сработало, что нежелательно.

## 8. Итоги и ключевые моменты

- **Абстрактные классы** (`abc.ABC`, `@abstractmethod`) – для явного задания интерфейса и общей логики. Экземпляр создать нельзя.
- **Протоколы** (`typing.Protocol`) – для структурной типизации (статической утиной типизации). Не требуют наследования.
- **`@runtime_checkable`** – позволяет использовать `isinstance` с протоколами (проверка наличия методов).
- **Выбор подхода**:
  - Нужна иерархия с общей реализацией → абстрактный класс.
  - Нужен гибкий интерфейс для независимых классов → протокол.
  - Можно комбинировать.

### 🧠 Проверь себя

**Вопрос 1:** Можно ли создать экземпляр класса, который наследуется от `ABC`, но не переопределил абстрактный метод?

**Вопрос 2:** Чем протокол отличается от абстрактного класса?

**Вопрос 3:** Зачем нужен `@runtime_checkable`?

<details>
<summary>Ответы</summary>

1. Нет, интерпретатор выдаст `TypeError`.
2. Протокол не требует наследования, не может содержать реализацию методов, используется для статической проверки структурного соответствия.
3. Чтобы можно было использовать `isinstance(obj, Protocol)` во время выполнения (проверяется наличие методов, определённых в протоколе).

</details>

---

## 📖 Домашнее задание

1. **Абстрактный класс `FileProcessor`**  
   Создайте абстрактный класс с методами `read()`, `process(data)`, `write(data)`. Реализуйте два наследника: `TxtProcessor` (читает/пишет текст) и `JsonProcessor` (работает с JSON).

2. **Протокол `Comparable`**  
   Определите протокол с методом `__lt__(self, other)`. Напишите функцию `sort_objects`, которая принимает список объектов, поддерживающих сравнение, и возвращает отсортированный список. Проверьте на списке чисел и на списке строк.

3. **Комбинированное задание**  
   Создайте протокол `Drawable` (метод `draw()`) и абстрактный класс `Shape` (методы `area`, `perimeter`). Реализуйте классы `Circle`, `Rectangle`, которые наследуют `Shape` и поддерживают `Drawable`. Напишите функцию, которая рисует список `Drawable` объектов и выводит их площади.